# OpenGeoStreams library check

Run the cells **top to bottom, one at a time** (Shift+Enter). Each section calls one part of the
library and shows its result, so you can see whether it runs and what it returns.

- Interactive Plotly charts appear under their cell; the Folium map appears as an embedded map.
- Cells marked **needs internet** fetch weather or elevation data; they print a message instead of failing when offline.
- Files written by this notebook go to the `outputs/` folder in the project root.

## 1. Setup

In [1]:
import os
from pathlib import Path
from pprint import pprint

# Run from the project root whether the notebook was opened from examples/ or the root.
if Path.cwd().name == "examples":
    os.chdir(Path.cwd().parent)
ROOT = Path.cwd()
OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)

import pandas as pd
import opengeostreams as ogs
from opengeostreams.opengeostreams import network, qualitycheck as qc

CSV = ROOT / "Datasets" / "consolidated_dataset.csv"
print("Project root:", ROOT)
print("CSV found:", CSV.is_file())

Project root: d:\Projects\N-choe\CURATED DATA
CSV found: True


## 2. Load a CSV: `ogs.load_csv`

In [2]:
ds = ogs.load_csv(CSV, rows="all")
print(type(ds).__name__)
print("loaded rows:", ds.loaded_rows, "| total rows:", ds.total_rows, "| more rows left:", ds.has_more)

RiverDataset
loaded rows: 1560 | total rows: 1560 | more rows left: False


Loading only part of a file: `rows=100` reads the first 100 rows.

In [6]:
partial = ogs.load_csv(CSV, rows=1560)
print("loaded rows:", partial.loaded_rows, "| remaining:", partial.remaining_rows, "| has_more:", partial.has_more)

loaded rows: 1560 | remaining: 0 | has_more: False


## 3. Look at the data: `head`, `data`, `summary`

In [7]:
ds.head(10)

,SrNo,Parameter,Unit,Date,Value,Data Source,Location,Latitude,Longitude
0,1,pH,-,"[01-01-2018, 31-01-2018]",6.6,CPCB 2018,NCM03 - Sector 83,30.658143,76.724407
1,2,Conductivity,µs/cm,"[01-01-2018, 31-01-2018]",644,CPCB 2018,NCM03 - Sector 83,30.658143,76.724407
2,3,DO,mg/l,"[01-01-2018, 31-01-2018]",,CPCB 2018,NCM03 - Sector 83,30.658143,76.724407
3,4,COD,mg/l,"[01-01-2018, 31-01-2018]",261,CPCB 2018,NCM03 - Sector 83,30.658143,76.724407
4,5,BOD,mg/l,"[01-01-2018, 31-01-2018]",161,CPCB 2018,NCM03 - Sector 83,30.658143,76.724407
5,6,NO3-N,mg/l,"[01-01-2018, 31-01-2018]",5.9,CPCB 2018,NCM03 - Sector 83,30.658143,76.724407
6,7,NH3-N,mg/l,"[01-01-2018, 31-01-2018]",23,CPCB 2018,NCM03 - Sector 83,30.658143,76.724407
7,8,Phosphate,mg/l,"[01-01-2018, 31-01-2018]",2.55,CPCB 2018,NCM03 - Sector 83,30.658143,76.724407
8,9,TSS,mg/l,"[01-01-2018, 31-01-2018]",100,CPCB 2018,NCM03 - Sector 83,30.658143,76.724407
9,10,TDS,mg/l,"[01-01-2018, 31-01-2018]",326,CPCB 2018,NCM03 - Sector 83,30.658143,76.724407


In [8]:
ds.data.dtypes

SrNo              int64
Parameter           str
Unit                str
Date                str
Value               str
Data Source         str
Location            str
Latitude        float64
Longitude       float64
NumericValue    float64
dtype: object

In [10]:
summary = ds.summary
for key in ("recordCount", "locationCount", "parameterCount", "mappedLocationCount",
            "unmappedLocationCount", "missingValueCount", "loadedRows", "totalRows"):
    print(f"{key:24} {summary.get(key)}")
print("\nmerged station labels:", len(summary["mergedLocations"]))
pprint(summary["mergedLocations"][:3])

recordCount              1560
locationCount            13
parameterCount           51
mappedLocationCount      13
unmappedLocationCount    0
missingValueCount        512
loadedRows               1560
totalRows                1560

merged station labels: 1
[{'merged': ['Sector 83'], 'name': 'NCM03 - Sector 83'}]


## 4. Statistics: `describe`

In [11]:
ds.describe()

,count,mean,std,min,25%,50%,75%,max
Parameter,,,,,,,,
BOD,70.0,4.727143e+01,3.669021e+01,3.000000,1.325000e+01,48.00,7.200000e+01,161.00
BOD (Max),3.0,1.710000e+02,4.757100e+01,118.000000,1.515000e+02,185.00,1.975000e+02,210.00
BOD (Min),2.0,2.550000e+00,6.363961e-01,2.100000,2.325000e+00,2.55,2.775000e+00,3.00
Boron(B),20.0,1.180000e-01,6.955498e-02,0.000000,8.000000e-02,0.14,1.500000e-01,0.21
COD,64.0,1.216406e+02,7.144037e+01,11.000000,6.025000e+01,118.00,1.730000e+02,261.00
Ca as CaCO3,24.0,1.240000e+02,1.149669e+01,100.000000,1.195000e+02,121.00,1.305000e+02,142.00
Chloride,31.0,4.874194e+01,1.765213e+01,29.000000,4.150000e+01,45.00,4.700000e+01,123.00
Colour,31.0,1.870968e+01,1.117794e+01,0.000000,2.000000e+01,20.00,3.000000e+01,30.00
Conductivity,40.0,6.927500e+02,1.553357e+02,324.000000,6.160000e+02,663.00,6.990000e+02,1220.00


In [12]:
ds.describe("BOD")

,count,mean,std,min,25%,50%,75%,max
Parameter,,,,,,,,
BOD,70.0,47.271429,36.690212,3.0,13.25,48.0,72.0,161.0


## 5. Data quality: `coordinate_conflicts`

In [13]:
conflicts = ds.coordinate_conflicts()
print(len(conflicts), "station(s) with positions more than 500 m apart")
pprint(conflicts[:3])

0 station(s) with positions more than 500 m apart
[]


## 6. Build and combine datasets: `from_dataframe`, `concat`

In [14]:
frame = ds.data.copy()
first_half = ogs.from_dataframe(frame.iloc[: len(frame) // 2], name="first-half")
second_half = frame.iloc[len(frame) // 2 :]
combined = ogs.concat([first_half, second_half], name="combined.csv")
print("first half:", first_half.loaded_rows, "| combined:", combined.loaded_rows, "| original:", ds.loaded_rows)

first half: 780 | combined: 1560 | original: 1560


## 7. Export: `to_csv`

In [15]:
path = ds.to_csv(OUTPUT / "standardised_dataset.csv")
print("written:", path)
pd.read_csv(path).head(3)

written: D:\Projects\N-choe\CURATED DATA\outputs\standardised_dataset.csv


,SrNo,Parameter,Unit,Date,Value,Data Source,Location,Latitude,Longitude
0,1,pH,-,"[01-01-2018, 31-01-2018]",6.6,CPCB 2018,NCM03 - Sector 83,30.658143,76.724407
1,2,Conductivity,µs/cm,"[01-01-2018, 31-01-2018]",644,CPCB 2018,NCM03 - Sector 83,30.658143,76.724407
2,3,DO,mg/l,"[01-01-2018, 31-01-2018]",NaN,CPCB 2018,NCM03 - Sector 83,30.658143,76.724407


## 8. Choose what to plot

The cells below use one parameter and the station with the most readings of it. Change `PARAMETER` to try another.

In [16]:
PARAMETER = "BOD"
counts = ds.data.loc[ds.data["Parameter"] == PARAMETER, "Location"].value_counts()
LOCATION = counts.index[0]
print("parameter:", PARAMETER, "| station:", LOCATION, f"({counts.iloc[0]} readings)")
print("all parameters:", sorted(ds.data["Parameter"].unique()))

parameter: BOD | station: 3BRD (15 readings)
all parameters: ['BOD', 'BOD (Max)', 'BOD (Min)', 'Boron(B)', 'COD', 'Ca as CaCO3', 'Chloride', 'Colour', 'Conductivity', 'Conductivity (Max)', 'Conductivity (Min)', 'DO', 'DO (Max)', 'DO (Min)', 'FS', 'FS (Max)', 'FS (Min)', 'Faecal Coliform', 'Faecal Coliform (Max)', 'Faecal Coliform (Min)', 'Fixed Suspended Solids', 'Fluoride', 'Mg as CaCO3', 'NH3-N', 'NO2-N', 'NO3-N', 'NO3-N (Max)', 'NO3-N (Min)', 'P-Alkalinity', 'Phosphate', 'Phosphate-P', 'Potassium', 'SO4', 'Sodium', 'Sulphate', 'TDS', 'TH as CaCO3', 'TKN', 'TSS', 'Temperature', 'Temperature (Max)', 'Temperature (Min)', 'Total Coliform', 'Total Coliform (Max)', 'Total Coliform (Min)', 'Total alkalinity as CaCO3', 'Turbidity', 'VSS', 'pH', 'pH (Max)', 'pH (Min)']


## 9. Trend chart: `plot_trend` (precipitation part **needs internet**)

In [17]:
ds.plot_trend(parameter=PARAMETER, location=LOCATION)

## 10. Distribution (box plot): `plot_distribution`

In [ ]:
ds.plot_distribution(parameter='ph', grouping='month
', location=LOCATION)

ValueError: grouping must be 'year' or 'month'.

In [21]:
ds.plot_distribution(parameter=PARAMETER, grouping="month", location=LOCATION)

## 11. Drain / river matching: `channel_network`, `assign_channels` (elevation check **needs internet**)

In [ ]:
net = ds.channel_network()
print("channels with stations:", len(net["channels"]), "| unassigned stations:", len(net.get("unassigned", [])))
for channel in net["channels"]:
    print(f"- {channel.get('name')}: {len(channel.get('stations', []))} stations")

In [ ]:
ds.assign_channels()

In [ ]:
# Name helpers used for matching station labels to channel names.
print(network.parse_water_body("NCC02 - Exit of Chandigarh Sector 53 N-Choe"))
print(network.canonical_name("Budha Nallah"))
print(round(network.name_similarity("Sutlej", "Satluj"), 2))

## 12. Stream heatmap: `plot_stream`

In [ ]:
ds.plot_stream(parameter=PARAMETER)

In [ ]:
ds.plot_stream(parameter=PARAMETER, group_by_channel=True)

In [ ]:
ds.plot_stream(parameter=PARAMETER, channel=net["channels"][0]["id"])

## 13. Interpolation: `interpolate`, `plot`

In [ ]:
_, info = ds.interpolate()
pprint(info)
years = sorted({s["year"] for s in ds._interpolation["surfaces"]["yearly"] if s["parameter"] == PARAMETER})
print("years with a", PARAMETER, "surface:", years)

In [ ]:
ds.plot(parameter=PARAMETER)               # all years combined

In [ ]:
ds.plot(parameter=PARAMETER, year=years[-1])   # one year

## 14. Map: `plot_map` (saved as HTML and shown below)

In [ ]:
map_object = ds.plot_map(OUTPUT / "map.html", parameter=PARAMETER, open_browser=False)
print("saved:", map_object.path)
map_object

## 15. Water quality check: `qualitycheck`

In [ ]:
records = ds._dashboard_data["records"]
for use_case in ("human_consumption", "agriculture", "aquatic_life", "bathing"):
    results = qc.screen_stations(records, use_case)
    counts = pd.Series([r["status"] for r in results]).value_counts().to_dict()
    print(f"{use_case:18} {counts}")

In [ ]:
pd.DataFrame(qc.screen_stations(records, "bathing"))[["location", "sample", "label", "reasons"]]

In [ ]:
# One set of readings, and the safety scale for one value.
pprint(qc.fit_for_agriculture(pH=8.9, boron_mg_l=1.2, chloride_mg_l=200))
do_scale = qc.scale_definition("aquatic_life", "DO")
print(do_scale)
print("DO 3.5 mg/L is", qc.scale_status(do_scale, 3.5))

## 16. Save charts as PNG (needs `kaleido` and Chrome/Chromium)

In [ ]:
try:
    ds.plot_trend(save_path=OUTPUT / "trend.png", parameter=PARAMETER, location=LOCATION)
    ds.plot_distribution(save_path=OUTPUT / "distribution.png", parameter=PARAMETER)
    print("saved PNGs in", OUTPUT)
except Exception as error:
    print("PNG export not available:", error)

## 17. Station elevations: `fetch_elevations` (optional, needs Google Earth Engine set up)

In [ ]:
try:
    elevations = ds.fetch_elevations()
    display(elevations)
except Exception as error:
    print("Earth Engine not set up, skipped:", str(error).splitlines()[0])

## 18. Dashboard: `dashboard`

Starts the dashboard for this dataset and opens it in your browser. Run the next cell to stop it.
If a dashboard is already running on port 4173, stop it first or change the port below.

In [ ]:
server = ds.dashboard(port=4173)
print("dashboard running at", server.url)

In [ ]:
server.shutdown()
server.server_close()
print("dashboard stopped")